# Phase 2 — LM Judge: Classify MedQA Clarifying Questions

Classifies each clarifying question from the Phase 1 results as **EPISTEMIC** or **ALEATORIC**
using the LLM judge with clinical few-shot examples.

Reads: `outputs/medqa/phase1_results.csv`  
Writes: `outputs/medqa/phase1_cq_classified.csv`

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../../").resolve()))

# ── Dataset config ────────────────────────────────────────────────────────
DATASET              = "medqa"
ROOT                 = Path("../../").resolve()
PROMPTS_DIR          = ROOT / "prompts"  / DATASET
OUTPUTS_DIR          = ROOT / "outputs"  / DATASET

JUDGE_INSTRUCTION    = PROMPTS_DIR / "judge_instruction.txt"
PHASE1_RESULTS_PATH  = OUTPUTS_DIR / "phase1_results.csv"
OUTPUT_PATH          = OUTPUTS_DIR / "phase1_cq_classified.csv"

# ── Run config ────────────────────────────────────────────────────────────
MODEL_ID             = "gemini-2.5-flash"
REQUEST_INTERVAL     = 1.0
REGENERATE           = False   # set True to re-classify from scratch
# ─────────────────────────────────────────────────────────────────────────

import logging
import pandas as pd
from IPython.display import display, Markdown

from src.utils import load_dotenv
from src.providers import GeminiProvider
from src.judge import LLMJudge, CSVBatchClassifier, FewShotExample

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s — %(message)s",
    datefmt="%H:%M:%S",
)

load_dotenv(ROOT / ".env")
print("Environment loaded.")

## Few-Shot Examples

Clinical clarifying questions drawn from real encounter patterns.  
These are **not** from the MedQA dataset — no leakage risk.

In [ ]:
FEW_SHOT_EXAMPLES: list[FewShotExample] = [
    # ── ALEATORIC: ambiguous symptom description ──────────────────────────
    FewShotExample(
        input="When you say you've been feeling off lately, can you describe what that feels like?",
        expected_output="ALEATORIC",
        explanation=(
            "'Feeling off' is inherently ambiguous — fatigue, dizziness, low mood, "
            "nausea, or many other things. The uncertainty lies in the patient's "
            "own description, not in a missing clinical fact."
        ),
    ),
    # ── ALEATORIC: vague pain descriptor ─────────────────────────────────
    FewShotExample(
        input="What do you mean when you say the pain is 'bad' — is it sharp, dull, burning, or pressure-like?",
        expected_output="ALEATORIC",
        explanation=(
            "'Bad' has no clinical specificity. The same word could describe pain "
            "characters pointing to completely different diagnoses — the ambiguity "
            "is in the description itself, not a missing fact."
        ),
    ),
    # ── ALEATORIC: underspecified symptom with multiple valid readings ─────
    FewShotExample(
        input="You mentioned discomfort in your chest — do you mean pain, tightness, shortness of breath, or something else?",
        expected_output="ALEATORIC",
        explanation=(
            "'Chest discomfort' spans cardiac, respiratory, musculoskeletal, and GI "
            "causes. The model cannot assign a single meaning without the patient "
            "interpreting their own experience — multiple valid readings remain."
        ),
    ),
    # ── EPISTEMIC: specific diagnostic result ─────────────────────────────
    FewShotExample(
        input="Do you have your most recent chest X-ray results available?",
        expected_output="EPISTEMIC",
        explanation=(
            "The model understands the clinical picture but needs a concrete "
            "diagnostic result to reason further. The X-ray either exists or it "
            "doesn't — providing it fully closes the knowledge gap."
        ),
    ),
    # ── EPISTEMIC: specific measurable value ──────────────────────────────
    FewShotExample(
        input="What is your current temperature reading?",
        expected_output="EPISTEMIC",
        explanation=(
            "A specific measurable fact. The model is not confused about what the "
            "patient means — it simply needs the value to determine whether fever "
            "is present. The answer completely resolves the uncertainty."
        ),
    ),
    # ── EPISTEMIC: historical medical fact ────────────────────────────────
    FewShotExample(
        input="Have you been diagnosed with hypertension before?",
        expected_output="EPISTEMIC",
        explanation=(
            "A historical medical fact with a definitive yes or no answer. "
            "The model is filling a knowledge gap about the patient's background — "
            "the answer exists and fully resolves what the model needs to know."
        ),
    ),
]

print(f"Few-shot examples: {len(FEW_SHOT_EXAMPLES)}")
for ex in FEW_SHOT_EXAMPLES:
    print(f"  [{ex.expected_output}] {ex.input[:80]}")

## Smoke Test

In [ ]:
provider = GeminiProvider(model_id=MODEL_ID)

judge = LLMJudge(
    provider=provider,
    instructions_path=JUDGE_INSTRUCTION,
    few_shot_examples=FEW_SHOT_EXAMPLES,
    label_parser=lambda text: text.strip().upper(),
)

smoke = judge.evaluate("What is your current temperature reading?")
print(f"Smoke test → label={smoke.label}, error={smoke.error}")
assert smoke.label == "EPISTEMIC", f"Unexpected smoke test label: {smoke.label}"
print("Smoke test passed.")

## Load Phase 1 Results

In [ ]:
phase1_df = pd.read_csv(PHASE1_RESULTS_PATH)

# Keep only valid, unblocked rows with a real clarifying question
work_df = phase1_df[
    (~phase1_df["was_blocked"])
    & (phase1_df["clarifying_question"].notna())
    & (phase1_df["clarifying_question"].str.strip() != "")
    & (phase1_df["clarifying_question"] != "BLOCKED")
].copy()

print(f"Phase 1 rows total:  {len(phase1_df)}")
print(f"Usable CQs:          {len(work_df)}")
print()
display(work_df[["id", "ehr_summary", "clarifying_question"]].head(5))

## Classify Clarifying Questions

In [ ]:
# Save work_df as input CSV for the batch classifier
INPUT_CSV = OUTPUTS_DIR / "phase1_cq_input.csv"
work_df[["id", "clarifying_question"]].to_csv(INPUT_CSV, index=False)
print(f"Input CSV saved: {INPUT_CSV}")

if REGENERATE and OUTPUT_PATH.exists():
    OUTPUT_PATH.unlink()
    print("Deleted existing output — regenerating.")

classifier = CSVBatchClassifier(
    judge=judge,
    input_csv=INPUT_CSV,
    output_csv=OUTPUT_PATH,
    question_column="clarifying_question",
    id_column="id",
    delay_between_calls=REQUEST_INTERVAL,
)
classifier.run()
print(f"Classification complete → {OUTPUT_PATH}")

## Results Summary

In [ ]:
classified_df = pd.read_csv(OUTPUT_PATH)

# Merge with phase1 results for full context
merged = phase1_df.merge(
    classified_df[["id", "label", "latency_seconds", "error"]].rename(
        columns={"label": "cq_type", "latency_seconds": "judge_latency"}
    ),
    on="id", how="left",
)

valid_labels = {"EPISTEMIC", "ALEATORIC"}
classified = merged[merged["cq_type"].isin(valid_labels)]

print(f"Total classified: {len(classified_df)}")
print(f"Valid labels:     {len(classified)}")
print(f"Invalid/errors:   {len(classified_df) - len(classified)}")
print()
print("Label distribution:")
print(classified["cq_type"].value_counts().to_string())
print()

# Correctness by CQ type
print("Updated correctness by CQ type:")
display(
    classified.groupby("cq_type")[["is_correct_preliminary", "is_correct_updated", "confidence_delta"]]
    .agg({"is_correct_preliminary": "mean", "is_correct_updated": "mean", "confidence_delta": "mean"})
    .round(3)
)

print()
print("Sample classifications:")
display(classified[["id", "cq_type", "clarifying_question", "is_correct_updated", "confidence_delta"]].head(15))